In [1]:
import pandas as pd
import numpy as np
import altair as alt
import ipywidgets as widgets
from IPython.display import display, clear_output
import os
import re

# Частота дискретизации (можно подгружать из файла, если она разная)
fs = 500


In [2]:
def extract_number(name):
    numbers = re.findall(r'\d+', name)
    return int(numbers[0]) if numbers else float('inf')

def generate_title_text(path):
    return os.path.splitext(os.path.basename(path))[0]

def find_txt_files(folder_path):
    return [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.txt')]


In [4]:
def get_annotation_chart(lines, height=100):
    df = pd.DataFrame({
        'y': [i * 0.2 for i in range(len(lines))][::-1],
        'text': lines
    })
    return alt.Chart(df).mark_text(
        align='left',
        baseline='top',
        fontSize=12,
        color='gray'
    ).encode(
        y=alt.Y('y', axis=None),
        text='text:N'
    ).properties(
        width=230,
        height=height
    )

def get_comment_annotation():
    comments = [
        "Нормостеник, сердце немного под углом",
        "5 — Горизонтальный",
        "7 — Мягкие ткани",
        "9 — Вертикальный",
        "13 — Лёгкие",
        "16 — ЭКГ",
        "32 — Трансторакальный"
    ]
    return get_annotation_chart(comments)

def get_mean_annotation(df, columns):
    text_lines = [f'{col}: {df[col].mean():.2f}' for col in columns if col in df.columns]
    if not text_lines:
        return None
    return get_annotation_chart(["Среднее:"] + text_lines)


In [5]:
def load_file(file_path):
    global df, time, checkboxes, hidden_B_columns
    df = pd.read_csv(file_path, delimiter='\t', encoding='utf-8').dropna(axis=1, how='all')
    time = np.arange(len(df)) / fs

    checkboxes = []
    hidden_B_columns = []

    for col in df.columns:
        is_hidden = col.startswith('Б')
        checkboxes.append(widgets.Checkbox(value=not is_hidden, description=col))
        if is_hidden:
            hidden_B_columns.append(col)

    return widgets.VBox(checkboxes)

def get_long_df():
    selected_signals = [cb.description for cb in checkboxes if cb.value]
    long_data = []

    for col in selected_signals:
        if col not in df.columns:
            continue
        signal = df[col].copy()
        if invert_ecg_checkbox.value and col.startswith('ЭКГ'):
            signal = -signal
        long_data.append(pd.DataFrame({
            'Time': time,
            'Value': signal,
            'Signal': col
        }))

    return pd.concat(long_data, ignore_index=True) if long_data else pd.DataFrame()


In [ ]:
folder_path = '/content'  # Или твой путь
txt_files = sorted(find_txt_files(folder_path))
file_checkboxes = [
    widgets.Checkbox(value=True, description=generate_title_text(fp), layout=widgets.Layout(width='100%'))
    for fp in txt_files
]
files_box = widgets.VBox(
    file_checkboxes,
    layout=widgets.Layout(max_height='250px', overflow='auto', border='1px solid gray', padding='10px', width='100%')
)


In [ ]:
def plot_single_file(file_path):
    long_df = get_long_df()
    if long_df.empty:
        print("Нет выбранных сигналов.")
        return

    sorted_signals = sorted(long_df['Signal'].unique(), key=extract_number)
    selection = alt.selection_point(fields=['Signal'], bind='legend')

    chart = alt.Chart(long_df).mark_line().encode(
        x='Time',
        y='Value',
        color=alt.Color('Signal:N', sort=sorted_signals),
        opacity=alt.condition(selection, alt.value(1), alt.value(0.1))
    ).add_params(selection).properties(
        width=600,
        height=400,
        title=generate_title_text(file_path)
    ).interactive()

    annotation = get_comment_annotation()
    mean_annot = get_mean_annotation(df, hidden_B_columns)
    side_panel = alt.vconcat(annotation, mean_annot) if mean_annot is not None else annotation

    chart_final = alt.hconcat(chart, side_panel).resolve_legend(color="independent").configure_view(stroke=None)

    clear_output(wait=True)
    display(file_dropdown, checkbox_box, invert_ecg_checkbox, altair_button)
    display(chart_final)


In [ ]:
def get_long_df_from_df(df_local, time_local, selected_signals, invert_ecg):
    long_data = []
    for col in selected_signals:
        if col not in df_local.columns:
            continue
        signal = df_local[col].copy()
        if invert_ecg and col.startswith('ЭКГ'):
            signal = -signal
        long_data.append(pd.DataFrame({
            'Time': time_local,
            'Value': signal,
            'Signal': col
        }))
    return pd.concat(long_data, ignore_index=True) if long_data else pd.DataFrame()

def plot_multiple_files(selected_files, selected_signals, invert_ecg):
    for file_path in selected_files:
        print(f"📄 Файл: {generate_title_text(file_path)}")
        try:
            df_local = pd.read_csv(file_path, delimiter='\t', encoding='utf-8').dropna(axis=1, how='all')
            time_local = np.arange(len(df_local)) / fs
            long_df = get_long_df_from_df(df_local, time_local, selected_signals, invert_ecg)
            if long_df.empty:
                print("⚠️ Нет данных для выбранных сигналов.")
                continue

            sorted_signals = sorted(long_df['Signal'].unique(), key=extract_number)
            selection = alt.selection_point(fields=['Signal'], bind='legend')

            chart = alt.Chart(long_df).mark_line().encode(
                x='Time',
                y='Value',
                color=alt.Color('Signal:N', sort=sorted_signals),
                opacity=alt.condition(selection, alt.value(1), alt.value(0.1))
            ).add_params(selection).properties(
                width=600,
                height=400,
                title=generate_title_text(file_path)
            ).interactive()

            hidden_B_local = [col for col in df_local.columns if col.startswith('Б')]
            annotation = get_comment_annotation()
            mean_annot = get_mean_annotation(df_local, hidden_B_local)
            side_panel = alt.vconcat(annotation, mean_annot) if mean_annot is not None else annotation

            chart_final = alt.hconcat(chart, side_panel).resolve_legend(color="independent").configure_view(stroke=None)
            display(chart_final)

        except Exception as e:
            print(f"❌ Ошибка при обработке {file_path}: {e}")


In [ ]:
# Виджеты
file_dropdown = widgets.Dropdown(options=txt_files, value=txt_files[0], description='Файл:')
invert_ecg_checkbox = widgets.Checkbox(value=False, description='Инвертировать ЭКГ')
altair_button = widgets.Button(description='Построить график')
build_all_filtered_button = widgets.Button(description='Показать все выбранные')

# Колбэки
def refresh_single_plot(path):
    global checkbox_box
    checkbox_box = load_file(path)
    clear_output(wait=True)
    display(file_dropdown, checkbox_box, invert_ecg_checkbox, altair_button)

file_dropdown.observe(lambda ch: refresh_single_plot(ch['new']), names='value')
altair_button.on_click(lambda _: plot_single_file(file_dropdown.value))
build_all_filtered_button.on_click(lambda _: plot_multiple_files(
    [fp for fp, cb in zip(txt_files, file_checkboxes) if cb.value],
    [cb.description for cb in checkboxes if cb.value],
    invert_ecg_checkbox.value
))

# Стартовая загрузка
checkbox_box = load_file(file_dropdown.value)

# Отображение
display(file_dropdown, checkbox_box, invert_ecg_checkbox, altair_button)
display(widgets.HTML(value="<b>Выберите файлы для отображения:</b>"))
display(files_box)
display(build_all_filtered_button)
